# Balanced MERT vs CultureMERT Raga Benchmark

This is the recommended project notebook. It runs a compact but stronger experiment than the first quick test:

- 6 well-supported Saraga Carnatic ragas
- 6 original tracks per raga
- train/validation/test split by track, never by clip
- 6 evenly spaced 8-second clips per track
- all-layer frozen probes for MERT-95M and CultureMERT-95M
- clip and track accuracy, macro F1, top-3, confusion matrices and cluster metrics

Expected runtime on a T4 is roughly 45-90 minutes after the dataset is available. The first Saraga download may take longer.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU from Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))

## Optional persistence

Google Drive keeps the 2 GB Saraga download and embedding cache if Colab disconnects. Set `USE_DRIVE=False` for a disposable run.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_HOME = '/content/drive/MyDrive/mert_raga_project/saraga_carnatic'
    OUTPUT_DIR = '/content/drive/MyDrive/mert_raga_project/balanced_benchmark'
else:
    DATA_HOME = '/content/saraga_carnatic'
    OUTPUT_DIR = '/content/balanced_benchmark'

print('Data cache:', DATA_HOME)
print('Outputs:', OUTPUT_DIR)

## Install and clone

PyTorch is already installed by Colab. The Transformers version is pinned because newer releases can break MERT's custom model code.

In [ ]:
!pip -q install 'transformers==4.41.0' 'librosa>=0.10' soundfile numpy pandas scikit-learn matplotlib seaborn tqdm pyyaml mirdata joblib nnAudio

import os
if os.path.exists('/content/mert-raga-classification'):
    %cd /content/mert-raga-classification
    !git pull --ff-only
else:
    %cd /content
    !git clone https://github.com/propixx/mert-raga-classification.git
    %cd /content/mert-raga-classification

!python scripts/00_check_env.py

## Run the benchmark

The script caches clips and embeddings. If a run stops, rerun this cell and it will reuse completed work.

In [ ]:
!python scripts/10_balanced_benchmark.py \
  --dataset saraga_carnatic \
  --data-home "$DATA_HOME" \
  --output-dir "$OUTPUT_DIR" \
  --num-ragas 6 \
  --tracks-per-raga 6 \
  --segments-per-track 6 \
  --segment-seconds 8 \
  --batch-size 6

## Read the result

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display, Image

out = Path(OUTPUT_DIR)
display(Markdown((out / 'REPORT.md').read_text()))
display(Image(filename=str(out / 'figures' / 'dataset_distribution.png')))
display(Image(filename=str(out / 'figures' / 'mert_95m_layer_curve.png')))
display(Image(filename=str(out / 'figures' / 'culturemert_95m_layer_curve.png')))

## Download the compact result package

This zip excludes the large audio and embedding cache. It contains the report, metrics, figures and trained probes.

In [ ]:
from google.colab import files
result_zip = str(Path(OUTPUT_DIR) / 'mert_raga_benchmark_results.zip')
print(result_zip)
files.download(result_zip)